# Blind Spots of Qwen3.5-4B-Base

**Model**: [`Qwen/Qwen3.5-4B-Base`](https://huggingface.co/Qwen/Qwen3.5-4B-Base)  
**Type**: Pre-trained base model (not instruction-tuned)  
**Parameters**: ~4B  
**Released**: February 2026  
**License**: Apache 2.0

This notebook probes the model for systematic failure modes and blind spots across diverse reasoning categories.

## 1. Setup

In [3]:
# Install required packages
!pip install -q transformers>=4.47.0 accelerate torch bitsandbytes huggingface_hub
!pip install --upgrade transformers

^C
ERROR: Operation cancelled by user
  Using cached transformers-5.3.0-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached numpy-2.4.2-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached regex-2026.2.28-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached filelock-3.25.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.3.2-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
  Using cached c

In [ ]:
import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Load Model

Qwen3.5-4B-Base uses a hybrid architecture (Gated DeltaNet linear attention + sparse MoE + gated attention). We load it in bfloat16 to fit on a single T4/A10 GPU.

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-4B-Base"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model (this may take a few minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto",
)
model.eval()
print("Model loaded successfully.")

## 3. Generation Helper

In [ ]:
def generate(prompt: str, max_new_tokens: int = 80, temperature: float = 0.1) -> str:
    """Generate a completion for a given prompt using greedy-ish decoding."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=temperature,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Only return the newly generated tokens
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def probe(test_case: dict) -> dict:
    """Run a single test case and print a formatted result."""
    prompt = test_case["prompt"]
    model_output = generate(prompt)
    result = {**test_case, "model_output": model_output}
    print(f"--- [{test_case['id']}] {test_case['category']} ---")
    print(f"PROMPT   : {prompt}")
    print(f"EXPECTED : {test_case['expected_output']}")
    print(f"MODEL    : {model_output}")
    print()
    return result

## 4. Test Cases

12 diverse prompts across categories where language model base models are known to struggle:
arithmetic, character-level reasoning, cognitive reflection, spatial reasoning, logic,
commonsense physics, probabilistic reasoning, temporal arithmetic, factual myths,
multi-hop ordering, linguistic illusions, and Winograd schemas.

In [ ]:
TEST_CASES = [
    {
        "id": "tc01",
        "category": "arithmetic",
        "prompt": "Q: What is 347 multiplied by 28?\nA:",
        "expected_output": "9716",
        "explanation": "Multi-digit multiplication requires carrying. Base models often produce plausible-looking but incorrect numbers.",
    },
    {
        "id": "tc02",
        "category": "character_level_reasoning",
        "prompt": "Q: Count the number of times the letter 'r' appears in the word 'strawberry'.\nA:",
        "expected_output": "3",
        "explanation": "Tokenizers split words into subwords, obscuring individual character counts. Models routinely answer '2' for this word.",
    },
    {
        "id": "tc03",
        "category": "cognitive_reflection",
        "prompt": "Q: A bat and a ball together cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?\nA:",
        "expected_output": "$0.05",
        "explanation": "Classic Cognitive Reflection Test item. The intuitive (wrong) answer is $0.10. Requires suppressing the first plausible response.",
    },
    {
        "id": "tc04",
        "category": "spatial_reasoning",
        "prompt": "Q: A large wooden cube is painted red on all 6 faces and then cut into 27 equal smaller cubes. How many of the small cubes have exactly 2 red faces?\nA:",
        "expected_output": "12",
        "explanation": "Requires spatial mental rotation: 12 edge-cubes have exactly 2 painted faces. Models often confuse edge-cubes (2 faces) with corner-cubes (3 faces), answering 8.",
    },
    {
        "id": "tc05",
        "category": "formal_logic",
        "prompt": "Q: All roses are flowers. Some flowers fade quickly. Does it necessarily follow that some roses fade quickly?\nA:",
        "expected_output": "No",
        "explanation": "Invalid syllogism: the flowers that fade quickly may be a disjoint set from roses. Models often affirm this invalid inference due to plausible surface associations.",
    },
    {
        "id": "tc06",
        "category": "commonsense_physics",
        "prompt": "Q: Five birds are sitting on a fence. A hunter shoots and kills two of them. How many birds are left sitting on the fence?\nA:",
        "expected_output": "0",
        "explanation": "The gunshot scares the survivors away; none remain on the fence. The naive arithmetic answer is 3.",
    },
    {
        "id": "tc07",
        "category": "probabilistic_reasoning",
        "prompt": "Q: A rare disease affects 1 in 1,000 people. A diagnostic test has a 99% accuracy rate (false positive rate: 1%). A randomly selected person tests positive. What is the approximate probability they actually have the disease?\nA:",
        "expected_output": "Approximately 9% (about 1 in 11)",
        "explanation": "Bayes' theorem: P(disease|positive) ≈ 0.001*0.99 / (0.001*0.99 + 0.999*0.01) ≈ 9%. Models exhibit base-rate neglect, often answering 99%.",
    },
    {
        "id": "tc08",
        "category": "temporal_arithmetic",
        "prompt": "Q: If today is Wednesday, what day of the week will it be exactly 100 days from now?\nA:",
        "expected_output": "Friday",
        "explanation": "100 mod 7 = 2; Wednesday + 2 days = Friday. Models often miscalculate the modular arithmetic.",
    },
    {
        "id": "tc09",
        "category": "factual_misconception",
        "prompt": "Q: True or False: The Great Wall of China is clearly visible from space with the naked eye.\nA:",
        "expected_output": "False",
        "explanation": "A widespread myth; the Wall is too narrow to be seen from low Earth orbit without optical aids. Models trained on internet text absorb this misconception.",
    },
    {
        "id": "tc10",
        "category": "multi_hop_ordering",
        "prompt": "Q: Alex is older than Jamie. Jamie is older than Sam. Sam is older than Taylor. Who is the youngest person?\nA:",
        "expected_output": "Taylor",
        "explanation": "Requires chaining three transitive relations. Models sometimes short-circuit and report Sam or Jamie.",
    },
    {
        "id": "tc11",
        "category": "linguistic_illusion",
        "prompt": "Q: In the famous biblical story, how many animals of each kind did Moses take on the Ark?\nA:",
        "expected_output": "None — it was Noah who built the Ark, not Moses.",
        "explanation": "The Moses Illusion: the wrong proper noun is substituted and readers (and models) often miss it, answering 'two' instead of catching the error.",
    },
    {
        "id": "tc12",
        "category": "winograd_schema",
        "prompt": "Q: The trophy didn't fit in the suitcase because it was too big. What was too big?\nA:",
        "expected_output": "The trophy",
        "explanation": "Winograd schema requiring commonsense coreference resolution. 'It' refers to the trophy (the thing that wouldn't fit), but models sometimes resolve it to the suitcase.",
    },
]

## 5. Run All Probes

In [ ]:
results = [probe(tc) for tc in TEST_CASES]

## 6. Save Results to JSONL

In [ ]:
OUTPUT_FILE = "blind_spots_data.jsonl"

with open(OUTPUT_FILE, "w") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Saved {len(results)} records to {OUTPUT_FILE}")

## 7. (Optional) Upload Dataset to Hugging Face

After filling in the dataset README and verifying your results, you can push the dataset directly from Colab.

In [1]:
from huggingface_hub import HfApi, login
login()
api = HfApi()
api.create_repo("mohammedfirdouss/qwen35-4b-base-blind-spots", repo_type="dataset")
api.upload_folder(
    folder_path="dataset/",
    repo_id="mohammedfirdouss/qwen35-4b-base-blind-spots",
    repo_type="dataset",
)

ModuleNotFoundError: No module named 'huggingface_hub'